# 04 · Trayectorias de 7 puntos para el logo (S7)

**Historias:** US-02.6 (geometría base del logo) y SP-02.6. **Kernel:** `colibri-yolo` (no usa GPU).

Cada video de prueba se reduce a **7 puntos en total** sobre todos sus segmentos (D-12). Los huecos sin ave no son
recorrido: no reparten puntos ni se dibujan; la curva se corta y se marca el hueco.

| Método | Qué hace | Para qué sirve |
|---|---|---|
| `a_uniforme_tiempo` | 7 instantes equiespaciados en el tiempo **con ave** | ritmo de la visita |
| `b_salientes` | Visvalingam-Whyatt: conserva los puntos que más forma aportan | forma del recorrido |

Por los puntos de cada tramo pasa una Catmull-Rom centrípeta. El error de forma es la distancia media y máxima del
recorrido denso suavizado a esa curva. **Referencia:** `resultados/S7_trayectorias_2d/resumen_7pts.csv`.

**Entrada:** las trayectorias prudentes de S5. En modo `artefactos`, las congeladas en `resultados/S5_tracking_union_test/`;
en modo `recalcular`, las que acaba de escribir el notebook 03 en `runs/03/`.

**Abierto (CA-02.6.6):** qué método va al logo, o un híbrido. Ver `docs/decisiones.md`.

In [ ]:
%run 00_comun.ipynb
SRC = (P["resultados"] / "S5_tracking_union_test" / "trayectorias_prudente") if MODO == "artefactos" else (RUNS / "03" / "trayectorias_prudente")
OUT = RUNS / "04"; OUT.mkdir(parents=True, exist_ok=True)
REF = P["resultados"] / "S7_trayectorias_2d"
VIDS_TEST = videos(("test",))
print("trayectorias de", SRC)

## 1. Siete puntos, curvas y láminas

Cuerpo de `main()` del script de S7, con las rutas de `config.yaml`. Escribe por video y método un CSV de puntos
(`x_px`, `y_px`, normalizados, `es_puente_corregido`, `hueco_antes`), la figura con fondo y el trazo solo, y dos
láminas resumen: a escala libre y a la misma escala (cuadro completo 16:9).

In [ ]:
# Portado de video-val/resultados/trayectorias_7pts/trayectorias_7pts.py
def main():
    OUT.mkdir(parents=True, exist_ok=True)
    files = sorted(SRC.glob("trayectoria_*.csv"))
    fig, axes = plt.subplots(len(files), 2, figsize=(24, 27))
    fig_n, axes_n = plt.subplots(len(files), 2, figsize=(24, 27))
    summary = []
    for i, f in enumerate(files):
        key = next(k for k in NAMES if k in f.stem); name = NAMES[key]
        video = next(v["path"] for k_, v in VIDS_TEST.items() if key in k_)
        cap = cv2.VideoCapture(str(video)); fps = cap.get(cv2.CAP_PROP_FPS); W, H = int(cap.get(3)), int(cap.get(4))
        t = pd.read_csv(f).sort_values("frame").reset_index(drop=True)
        dense = np.vstack([smooth(s[["cx_px", "cy_px"]].to_numpy(), fps) for _, s in t.groupby("segment_id", sort=False)])
        bridge = t.es_puente.astype(bool).to_numpy()
        seg_ids = t.segment_id.to_numpy()
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(t.frame.median())); _, fr = cap.read(); cap.release()
        bg = cv2.cvtColor(fr, cv2.COLOR_BGR2RGB)
        presence_s = len(t) / fps
        for j, (m, fn) in enumerate((("a_uniforme_tiempo", uniform_time), ("b_salientes", visvalingam))):
            idx, moved = fix_bridges(fn(dense), bridge)
            P = dense[idx]
            sel = t.iloc[idx]
            curves = curves_by_stretch(P, seg_ids[idx])
            gaps = gaps_between(sel)
            e_mean, e_max = shape_error(dense, np.vstack(curves))
            size = float(np.hypot(*np.ptp(dense, axis=0)))
            summary.append(dict(video=name, metodo=m, segmentos=t.segment_id.nunique(), segmentos_con_punto=sel.segment_id.nunique(),
                                filas_densas=len(t), t_ini=round(float(t.t_s.iloc[0]), 2), t_fin=round(float(t.t_s.iloc[-1]), 2),
                                tiempo_con_ave_s=round(presence_s, 2), puntos_movidos=int(sum(moved)), huecos_entre_puntos=len(gaps),
                                error_medio_px=round(e_mean, 1), error_max_px=round(e_max, 1),
                                error_medio_pct_diagonal=round(100 * e_mean / size, 1)))
            hueco = [False] * len(idx)
            for k, _ in gaps:
                hueco[k + 1] = True
            pd.DataFrame(dict(punto=range(1, len(idx) + 1), frame=sel.frame.to_numpy(), t_s=sel.t_s.round(3).to_numpy(),
                              segment_id=sel.segment_id.to_numpy(), x_px=P[:, 0].round(1), y_px=P[:, 1].round(1),
                              x_norm=(P[:, 0] / W).round(4), y_norm=(P[:, 1] / H).round(4), es_puente_corregido=moved,
                              hueco_antes=hueco)).to_csv(OUT / f"{name}_{m}.csv", index=False)
            # con fondo
            f1, a1 = plt.subplots(figsize=(12, 12 * H / W))
            a1.imshow(bg, alpha=0.55)
            for sid in np.unique(seg_ids):
                d = dense[seg_ids == sid]
                a1.plot(d[:, 0], d[:, 1], color="0.3", lw=1)
            for c in curves:
                a1.plot(c[:, 0], c[:, 1], color="crimson", lw=2.5)
            a1.scatter(P[:, 0], P[:, 1], s=60, c="crimson", edgecolors="white", zorder=5)
            for k, (x, y) in enumerate(P, 1):
                a1.annotate(str(k), (x, y), xytext=(6, 6), textcoords="offset points", fontsize=11, weight="bold",
                            bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.85))
            mark_gaps(a1, P, gaps, color="black")
            a1.set_title(f"{name} | {METHODS[m]} | {sel.t_s.iloc[0]:.1f}-{sel.t_s.iloc[-1]:.1f} s | "
                         f"{len(gaps)} huecos | error medio {e_mean:.0f} px", fontsize=11); a1.axis("off")
            f1.savefig(OUT / f"{name}_{m}_fondo.png", dpi=110, bbox_inches="tight"); plt.close(f1)
            # solo trazo
            f2, a2 = plt.subplots(figsize=(10, 10))
            sketch(a2, curves, P, gaps)
            f2.savefig(OUT / f"{name}_{m}_trazo.png", dpi=110, bbox_inches="tight", facecolor="white"); plt.close(f2)
            sketch(axes[i, j], curves, P, gaps, f"{name} | {METHODS[m]} | {sel.t_s.iloc[0]:.1f}-{sel.t_s.iloc[-1]:.1f} s | "
                   f"{len(gaps)} huecos | error medio {e_mean:.0f} px ({100 * e_mean / size:.1f} %)", numbers=True)
            sketch_norm(axes_n[i, j], curves, P, gaps, W, H, f"{name} | {METHODS[m]} | {sel.t_s.iloc[0]:.1f}-{sel.t_s.iloc[-1]:.1f} s | {len(gaps)} huecos")
    fig.tight_layout()
    fig.savefig(OUT / "trayectorias_7pts.jpg", dpi=100, facecolor="white", pil_kwargs={"quality": 92}); plt.close(fig)
    fig_n.tight_layout()
    fig_n.savefig(OUT / "trayectorias_7pts_misma_escala.jpg", dpi=100, facecolor="white", pil_kwargs={"quality": 92}); plt.close(fig_n)
    S = pd.DataFrame(summary); S.to_csv(OUT / "resumen_7pts.csv", index=False)
    print(S.to_string(index=False))
main()
RES = pd.read_csv(OUT / "resumen_7pts.csv")

## 2. Comparación con S7

El resumen y cada CSV de puntos deben coincidir (posiciones con tolerancia de 1 px).

In [ ]:
comparar(RES, REF / "resumen_7pts.csv", ["video", "metodo"], ["puntos_movidos", "huecos_entre_puntos", "error_medio_px", "error_max_px"],
         tol=TOL["px"], nombre="resumen de 7 puntos frente a S7")
for f in sorted(OUT.glob("*_?_*.csv")):
    if (REF / f.name).exists():
        comparar(pd.read_csv(f), REF / f.name, ["punto"], ["frame", "x_px", "y_px"], tol=TOL["px"], nombre=f.stem)

In [ ]:
from IPython.display import Image as _Img
display(_Img(filename=str(OUT / "trayectorias_7pts_misma_escala.jpg"), width=900))

## Conclusiones (S7)

- **Incremento:** 7 puntos por video, con dos métodos, curvas y láminas listas para diseño.
- **Decisión D-12:** 7 puntos en total por video sobre todos los segmentos; los huecos no se dibujan.
- Los dos métodos dan errores de forma parecidos (6-9 % de la diagonal del recorrido); `salientes` baja el error
  **máximo** (DoJ1: 205 → 88 px), así que conserva mejor los giros.
- **Pendiente:** elegir método para E-03 (CA-02.6.6) junto con el brief del logo (US-04.1).